# 6. PPT — Reporte Diario (autónomo)

Versión autocontenida del armado del reporte diario. Las funciones de filtrado, cálculos de indicadores, cruces de cortes, gráficas y sustitución de marcadores están definidas en este cuaderno; no importa `config.py` ni módulos de `src/` o `1_funciones/`.

**Datos y reproducibilidad:** consume el snapshot del universo RNPDNO/BANCI ya procesado, además de los Excel congelados de referencia, MJ/RIPS y directorio de indicios. No vuelve a ejecutar los motores de matching. Si se actualizan las fuentes o reglas, primero se debe regenerar el snapshot mediante el pipeline oficial. El PPTX se fecha con el día de ejecución.

## 1. Importar librerías

In [ ]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from dataclasses import dataclass, replace
from datetime import date, datetime
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from IPython.display import Image, display

pd.set_option("display.max_columns", 100)
np.random.seed(0)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 2. Configurar entorno y reproducibilidad

In [ ]:
def encontrar_raiz(inicio: Path) -> Path:
    for p in (inicio.resolve(), *inicio.resolve().parents):
        if (p / "output" / "cache" / "universo").is_dir() and (p / "input" / "plantillas").is_dir():
            return p
    raise FileNotFoundError("No se localizó la raíz del proyecto (output/cache/universo e input/plantillas).")

RAIZ = encontrar_raiz(Path.cwd())
INPUT = RAIZ / "input"
OUTPUT = RAIZ / "output"
RESULTADOS = OUTPUT / "Resultados_4_Generacion_Indicadores"
CACHE_UNIVERSO = OUTPUT / "cache" / "universo" / "9132b089827c4e23"
PLANTILLA = INPUT / "plantillas" / "reporte_diario.pptx"
RUTA_GRAFICAS = RAIZ / "notebooks" / "output"
FUENTES_DIR = RAIZ / "0_fonts"
SALIDA_PPT = OUTPUT / "ppt_reporte_diario" / f"Reporte de personas desaparecidas - {date.today():%d.%m.%Y}.pptx"
FECHA_EMISION = date.today()

PERIODOS = {
    "min-max": (None, None), "min-01ene06": (None, "2006-01-01"),
    "01ene06-max": ("2006-01-01", None), "min-01oct24": (None, "2024-10-01"),
    "01oct24-max": ("2024-10-01", None), "01ago25-max": ("2025-08-01", None),
    "01oct24-01ago25": ("2024-10-01", "2025-08-01"),
    "01ago26-max": ("2026-08-01", None), "01jul26-max": ("2026-07-01", None),
}
ENTIDADES = {
    1:"Aguascalientes",2:"Baja California",3:"Baja California Sur",4:"Campeche",5:"Coahuila",6:"Colima",7:"Chiapas",8:"Chihuahua",
    9:"Ciudad de México",10:"Durango",11:"Guanajuato",12:"Guerrero",13:"Hidalgo",14:"Jalisco",15:"México",16:"Michoacán",
    17:"Morelos",18:"Nayarit",19:"Nuevo León",20:"Oaxaca",21:"Puebla",22:"Querétaro",23:"Quintana Roo",24:"San Luis Potosí",
    25:"Sinaloa",26:"Sonora",27:"Tabasco",28:"Tamaulipas",29:"Tlaxcala",30:"Veracruz",31:"Yucatán",32:"Zacatecas",33:"Nacido en el Extranjero"
}
BASES = {"rn":("rn",None),"ba":("ba",None),
         "rn-no-ba":("rn",("in_banci_derivada_rnpdno_banci",False)),
         "rn-si-ba":("rn",("in_banci_derivada_rnpdno_banci",True)),
         "ba-no-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",False)),
         "ba-si-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",True))}
DIMS = {"rn":{"fecha":"fecha_hechos_modificada_derivada_rnpdno","cve_ent":"idestado_derivada_rnpdno","entidad":"nombre_entidad_derivada_rnpdno","estatus_loc":"estatus_loc_derivada_rnpdno"},
        "ba":{"fecha":"fha_de_hchos_derivada_banci","cve_ent":"cve_entidad_derivada_banci","entidad":"nombre_entidad_derivada_banci","estatus_loc":"estatus_loc_derivada_banci"}}
NEGACIONES={"sdel":"cdel","sind":"cind","scurp":"ccurp","cisf":"sisf","sfvac":"cfvac","sfval":"cfval"}
CONSTANTES={"rn":{},"ba":{"cind":True,"cfval":True,"cfvac":False,"fis":True,"com":False,"por":False}}
BANDERAS_ETL={"rn":{"ccurp":"curp_vic_valida_derivada_rnpdno"},"ba":{"ccurp":"curp_valida_derivada_banci"}}

# Archivos de cortes, MJ/RIPS y directorio de indicios.
CORTES = {
 "reg_rn_25mar26_des_sisf":(INPUT/"datos_marzo/reg_rn_25mar26_des_sisf.xlsx","fub"),
 "reg_rn_25mar26_des_cisf_cind":(INPUT/"datos_marzo/reg_rn_25mar26_des_cisf_cind.xlsx","fub"),
 "reg_rn_25mar26_des_cisf_sind":(INPUT/"datos_marzo/reg_rn_25mar26_des_cisf_sind.xlsx","fub"),
 "reg_rn_01jul26_des_sisf":(INPUT/"datos_julio/reg_rn_01jul26_des_sisf.xlsx","fub_rnpdno"),
 "reg_rn_01jul26_des_cisf_cind":(INPUT/"datos_julio/reg_rn_01jul26_des_cisf_cind.xlsx","fub_rnpdno"),
 "reg_rn_01jul26_des_cisf_sind":(INPUT/"datos_julio/reg_rn_01jul26_des_cisf_sind.xlsx","fub_rnpdno"),
}
SUFIJOS_CORTE={k:("_25mar26" if "25mar26" in k else "_01jul26") for k in CORTES}
DIRECTORIO_INDICIOS=INPUT/"domicilios_y_numeros_contacto/base_combinada_con_domicilio_y_numeros_de_contacto_24062026.xlsx"
MJ_RIPS_FILE=INPUT/"mj/MJ y RIPS (1062).xlsx"
CATALOGO_ENTIDADES_LAMINA=["Estado de México","Tamaulipas","Michoacán","Sinaloa","Guerrero","Veracruz","Ciudad de México","Sonora","Baja California","Zacatecas","Puebla","Nayarit","Chiapas","Tabasco","Hidalgo","San Luis Potosí","Quintana Roo","Morelos","Colima","Baja California Sur","Oaxaca","Yucatán","Tlaxcala","Campeche"]

RUTA_GRAFICAS.mkdir(parents=True,exist_ok=True)
RESULTADOS.mkdir(parents=True,exist_ok=True)
print("Raíz:",RAIZ,"| emisión:",FECHA_EMISION,"| caché:",CACHE_UNIVERSO)

## 3. Definir funciones auxiliares

In [ ]:
@dataclass(frozen=True, order=True)
class Slice:
    base: str="rn"; periodo: str="min-max"; estatus: str="all"; subfiltro: str="all"; estado: Optional[int]=None
    def __post_init__(self):
        if self.base not in BASES: raise ValueError(f"Base desconocida: {self.base}")
        if self.estatus not in {"all","des","loc"}: raise ValueError(f"Estatus desconocido: {self.estatus}")
        if self.periodo not in PERIODOS: raise ValueError(f"Período desconocido: {self.periodo}")
        if self.estado is not None and self.estado not in ENTIDADES: raise ValueError(f"Entidad desconocida: {self.estado}")
    @property
    def nombre(self):
        p=["reg",self.base,self.periodo]
        if self.estatus!="all": p.append(self.estatus)
        if self.subfiltro!="all": p.append(self.subfiltro)
        if self.estado is not None: p.append(f"ent{self.estado:02d}")
        return "_".join(p)
    @property
    def entidad_texto(self): return ENTIDADES.get(self.estado,"Nacional") if self.estado else "Nacional"
    @property
    def padre(self):
        if self.estado is not None: return replace(self,estado=None)
        if self.subfiltro!="all":
            t=self.subfiltro.split("_")[:-1]; return replace(self,subfiltro="_".join(t) if t else "all")
        if self.estatus!="all": return replace(self,estatus="all")
        return replace(self,periodo="min-max")
    def con(self,**kwargs): return replace(self,**kwargs)
    def por_entidad(self,cve): return self.con(estado=cve)


def expandir_entidades(base, entidades=None):
    return [base.por_entidad(i) for i in (entidades or sorted(ENTIDADES))]


def col_bandera(df,token):
    cols=[c for c in df.columns if c.startswith(f"is_{token}_derivada_")]
    if len(cols)!=1: raise KeyError(f"Bandera {token!r}: {cols}")
    return cols[0]


class Universo:
    """Acceso local al snapshot con la semántica de rebanado del pipeline."""
    def __init__(self,rn,ba,meta,auditoria=None): self.rn,self.ba,self.meta,self.auditoria=rn,ba,meta,auditoria; self._cache={}
    @classmethod
    def cargar(cls,ruta):
        faltan=[str(ruta/n) for n in ("rn.parquet","ba.parquet","meta.json") if not (ruta/n).is_file()]
        if faltan: raise FileNotFoundError("Snapshot incompleto: "+", ".join(faltan))
        rn=pd.read_parquet(ruta/"rn.parquet"); ba=pd.read_parquet(ruta/"ba.parquet"); meta=json.loads((ruta/"meta.json").read_text(encoding="utf-8"))
        if len(rn)!=meta.get("n_rn") or len(ba)!=meta.get("n_ba"): raise ValueError("El snapshot no coincide con meta.json")
        ap=ruta/"auditoria.csv"; aud=pd.read_csv(ap) if ap.exists() else None
        return cls(rn,ba,meta,aud)
    @staticmethod
    def frame(s): return BASES[s.base][0] if isinstance(s,Slice) else s
    def columna(self,s,dim): return DIMS[self.frame(s)][dim]
    def columna_bandera(self,s,token):
        f=self.frame(s); d=self.rn if f=="rn" else self.ba
        col=BANDERAS_ETL[f].get(token)
        if col and col in d: return col
        return col_bandera(d,token)
    def _periodo(self,f,p):
        k=(f,p)
        if k in self._cache:return self._cache[k]
        d=self.rn if f=="rn" else self.ba; fechas=pd.to_datetime(d[self.columna(f,"fecha")],errors="coerce")
        ini,fin=PERIODOS[p]; nulas=fechas.isna().to_numpy(); m=np.ones(len(d),dtype=bool)
        if ini is not None:m &= fechas.ge(pd.Timestamp(ini)).to_numpy()|nulas
        if fin is not None:m &= fechas.lt(pd.Timestamp(fin)).to_numpy()|nulas
        self._cache[k]=m; return m
    def _bandera(self,f,d,t,s):
        base=NEGACIONES.get(t,t)
        if base in CONSTANTES[f]: v=np.full(len(d),CONSTANTES[f][base],dtype=bool)
        else:v=d[self.columna_bandera(f,base)].fillna(False).astype(bool).to_numpy()
        return ~v if t in NEGACIONES else v
    def _frame_mascara(self,s):
        f,pertenece=BASES[s.base]; d=self.rn if f=="rn" else self.ba; m=self._periodo(f,s.periodo).copy()
        if pertenece:
            c,v=pertenece; b=d[c].fillna(False).to_numpy(dtype=bool); m &= b if v else ~b
        if s.estatus!="all":m &= d[self.columna(s,"estatus_loc")].eq(s.estatus).fillna(False).to_numpy()
        if s.subfiltro!="all":
            for t in s.subfiltro.split("_"):m &= self._bandera(f,d,t,s)
        if s.estado is not None:m &= d[self.columna(s,"cve_ent")].eq(s.estado).fillna(False).to_numpy()
        return d,m
    def contar(self,s):return int(self._frame_mascara(s)[1].sum())
    def slice(self,s):
        d,m=self._frame_mascara(s);return d.loc[m]
    def dataframes(self,ss):return {s.nombre:self.slice(s) for s in ss}
    def rango_fechas(self,s):
        f=self.frame(s); ini,fin=PERIODOS[s.periodo]; x=pd.to_datetime((self.rn if f=="rn" else self.ba)[self.columna(f,"fecha")],errors="coerce"); a,b=x.min(),x.max()
        return (a if ini is None else max(pd.Timestamp(ini),a), b if fin is None else min(pd.Timestamp(fin)-pd.Timedelta(days=1),b))
    def indicadores(self,ss,porcentajes=None,extra=None):
        out={s.nombre:self.contar(s) for s in ss}
        for item in porcentajes or []:
            s,d=(item if isinstance(item,tuple) else (item,None)); den=self.contar(d or s.padre); n=self.contar(s)
            out[f"pct_{s.nombre}"]=f"{100*n/den:.1f}%" if den else "n/d"
        out["entidad_texto"]="Nacional"
        for base,k in (("rn","fecha_corte_rnpdno"),("ba","fecha_corte_banci")):
            for formato in ("dia_mes_anio","mes_anio","anio"):out[f"fha_{base}_{formato}"]=fecha_es(self.meta.get(k),formato) if self.meta.get(k) else "Sin Fecha"
        hoy=date.today().isoformat()
        for formato in ("dia_mes_anio","dia_mes","mes_anio","anio"):out[f"fha_hoy_{formato}"]=fecha_es(hoy,formato)
        out.update(extra or {});return out
    def resumen(self):return pd.DataFrame([{"base":b,"registros":self.contar(Slice(b))} for b in BASES])


def fecha_es(fecha,partes="dia_mes_anio"):
    if not fecha or not isinstance(fecha,str):return fecha
    t=fecha.split("-")
    if len(t)!=3:return fecha
    y,m,d=t; meses={"01":"enero","02":"febrero","03":"marzo","04":"abril","05":"mayo","06":"junio","07":"julio","08":"agosto","09":"septiembre","10":"octubre","11":"noviembre","12":"diciembre"}
    v={"dia":str(int(d)),"mes":meses.get(m,m),"anio":y}; return " de ".join(v[x] for x in partes.split("_"))


def agrupar_por_mes(df,col_fecha,desde=None,hasta=None):
    f=pd.to_datetime(df[col_fecha],errors="coerce"); m=f.dropna().dt.to_period("M").dt.to_timestamp(); s=m.value_counts().sort_index()
    extremos=[pd.Timestamp(x).to_period("M").to_timestamp() for x in (desde,hasta) if x is not None]
    if len(s):extremos += [s.index.min(),s.index.max()]
    idx=pd.date_range(min(extremos),max(extremos),freq="MS") if extremos else pd.DatetimeIndex([])
    return pd.DataFrame({"mes":idx,"registros":s.reindex(idx,fill_value=0).astype(int).to_numpy()})


def porcentajes_particion(u,partes,denominador,sufijo="",decimales=1):
    ns=[u.contar(s) for s in partes]; total=u.contar(denominador)
    if sum(ns)!=total:raise ValueError(f"Partes no suman {denominador.nombre}: {ns} != {total}")
    keys=[f"pct_{s.nombre}{sufijo}" for s in partes]
    if not total:return dict.fromkeys(keys,"n/d")
    escala=100*10**decimales; cuotas=[divmod(escala*n,total) for n in ns]; un=[q for q,r in cuotas]; rs=[r for q,r in cuotas]
    for i in sorted(range(len(ns)),key=rs.__getitem__,reverse=True)[:escala-sum(un)]:un[i]+=1
    return {k:f"{n/10**decimales:.{decimales}f}%" for k,n in zip(keys,un)}


def _normaliza_header(x):return re.sub(r"[\s._\-]+","",unicodedata.normalize("NFD",str(x)).encode("ascii","ignore").decode()).lower().strip()
def _col_excel(path,nombre):
    cols=pd.read_excel(path,nrows=0).columns; target=_normaliza_header(nombre)
    for c in cols:
        if _normaliza_header(c)==target:return c
    raise KeyError(f"{path.name}: no existe columna {nombre}; columnas={list(cols)}")
def fubs_corte(nombre,verbose=True):
    path,col=CORTES[nombre]
    if not path.is_file():raise FileNotFoundError(f"Falta corte: {path}")
    real=_col_excel(path,col); df=pd.read_excel(path,dtype={real:str}); vals=df[real]; ids={str(v).strip() for v in vals.dropna() if str(v).strip()}; sin=int(vals.isna().sum()+vals.astype("string").str.strip().eq("").sum())
    if verbose:print(f"  📄 {nombre}: {len(ids):,} registros ({path.name})"+(f" (+{sin:,} sin fub)" if sin else ""))
    return ids,sin

def diferencias_vs_corte(vars_ppt,mapa,verbose=False):
    o={}
    for actual,corte in mapa.items():
        n=int(vars_ppt[actual]); nc=len(fubs_corte(corte,verbose)); k=f"{actual}-{corte}"; o[k]=n-nc;o[corte]=nc;o[f"pct_{k}"]=f"{100*(n-nc)/nc:.1f}%" if nc else "n/d"
    return o
def deltas_vs_corte(u,mapa,sufijo="",verbose=False):
    o={}
    for s,corte in mapa.items():
        act={str(x).strip() for x in u.slice(s)["fub_rnpdno"].dropna() if str(x).strip()}; prev,sinf=fubs_corte(corte,verbose); sinh=len(u.slice(s))-len(act)
        o[f"{s.nombre}_ddes{sufijo}"]=len(act-prev);o[f"{s.nombre}_dloc{sufijo}"]=len(prev-act)
        if sinf+sinh:o[f"{s.nombre}_dfub{sufijo}"]=sinf+sinh
    return o

## 4. Preparar datos de entrada y definir los indicadores

In [ ]:
u=Universo.cargar(CACHE_UNIVERSO)
if not PLANTILLA.is_file():raise FileNotFoundError(f"Falta plantilla de presentación: {PLANTILLA}")
print(f"Snapshot: {u.meta.get('generado')} | RNPDNO {len(u.rn):,} | BANCI {len(u.ba):,} | FUBs matcheados {u.meta.get('n_fubs_matcheados',0):,}")
print(u.resumen().to_string(index=False))

# Marcadores de texto de la plantilla, con la nomenclatura original.
CONTEOS=[
 Slice("rn"),Slice("rn",estatus="des"),Slice("rn",estatus="loc"),
 *[Slice("rn",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
 Slice("rn","01oct24-max"),Slice("rn","01oct24-max",estatus="des"),Slice("rn","01oct24-max",estatus="loc"),
 *[Slice("rn","01oct24-max",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
 Slice("rn","min-01ene06",estatus="des"),Slice("rn","01ene06-max",estatus="des"),
 Slice("rn","01ene06-max",estatus="des",subfiltro="sisf"),
 *[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("cisf","cisf_cind","cisf_sind")],
 *[Slice("rn","01oct24-max",estatus="des",subfiltro=t) for t in ("sisf","cisf_cind","cisf_sind")],
 *[Slice("rn-si-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("all","cpar","dfor")],
 Slice("rn-no-ba","01oct24-max",estatus="des"),
 *[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval","sfval_sfvac")],
 *[Slice("rn-no-ba",p,estatus="des",subfiltro=t) for p in ("01oct24-01ago25","01ago25-max") for t in ("cfval","sfval_sfvac")],
 *[Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro=t) for t in ("cfval","sfval_sfvac")],
 *[Slice("rn-no-ba",p,estatus="des",subfiltro=f"{folio}_{inst}") for p in ("01oct24-max","01ago26-max") for folio in ("cfval","sfval_sfvac") for inst in ("fis","com","por")],
 *[Slice("rn-no-ba",p,estatus="des",subfiltro="sfval_cfvac") for p in ("01oct24-max","01oct24-01ago25","01ago25-max")],
]
PORCENTAJES=[
 Slice("rn",estatus="des"),Slice("rn",estatus="loc"),Slice("rn","01oct24-max",estatus="des"),Slice("rn","01oct24-max",estatus="loc"),
 *[Slice("rn","01oct24-max",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
 *[Slice("rn",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
 *[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("sisf","cisf")],
 *[Slice("rn-si-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cpar","dfor")],
 *[(Slice("rn","01ene06-max",estatus="des",subfiltro=t),Slice("rn","01ene06-max",estatus="des")) for t in ("cisf_cind","cisf_sind")],
 *[(Slice("rn-no-ba",p,estatus="des",subfiltro=t),Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t)) for t in ("cfval","sfval_sfvac","sfval_cfvac") for p in ("01oct24-01ago25","01ago25-max")],
]
DEN_DES=Slice("rn","01oct24-max",estatus="des"); DEN_NOBA=Slice("rn-no-ba","01oct24-max",estatus="des"); DEN_SFVAL=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval")
PARTICIONES=[
 ("_v1",DEN_DES,[Slice("rn-si-ba","01oct24-max",estatus="des"),DEN_NOBA]),
 ("_v1",DEN_SFVAL,[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("sfval_sfvac","sfval_cfvac")]),
 ("_v2",DEN_DES,[Slice("rn-si-ba","01oct24-max",estatus="des"),*[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval")]]),
 ("_v3",DEN_NOBA,[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval_sfvac","sfval_cfvac")]),
]
MAPA_MARZO={"reg_rn_01ene06-max_des_sisf":"reg_rn_25mar26_des_sisf","reg_rn_01ene06-max_des_cisf_cind":"reg_rn_25mar26_des_cisf_cind","reg_rn_01ene06-max_des_cisf_sind":"reg_rn_25mar26_des_cisf_sind"}
MAPA_JULIO={"reg_rn_01ene06-max_des_sisf":"reg_rn_01jul26_des_sisf","reg_rn_01ene06-max_des_cisf_cind":"reg_rn_01jul26_des_cisf_cind","reg_rn_01ene06-max_des_cisf_sind":"reg_rn_01jul26_des_cisf_sind"}
SLICES_DELTA={Slice("rn","01ene06-max",estatus="des",subfiltro=t):m for t,m in zip(("sisf","cisf_cind","cisf_sind"),MAPA_MARZO.values())}
SLICES_DELTA_JULIO={Slice("rn","01ene06-max",estatus="des",subfiltro=t):m for t,m in zip(("sisf","cisf_cind","cisf_sind"),MAPA_JULIO.values())}

print(f"Rebanadas: {len(CONTEOS)} conteos | {len(PORCENTAJES)} porcentajes")

## 5. Procesar datos y calcular salidas

In [ ]:
def leer_excel_map(path, aliases, sheet=0, skiprows=0):
    if not path.is_file():raise FileNotFoundError(f"Falta insumo: {path}")
    df=pd.read_excel(path,sheet_name=sheet,skiprows=skiprows)
    norm={_normaliza_header(c):c for c in df.columns}; found={}
    for target,opts in aliases.items():
        opts=[opts] if isinstance(opts,str) else opts
        col=next((norm[_normaliza_header(x)] for x in opts if _normaliza_header(x) in norm),None)
        if col is None:raise KeyError(f"{path.name}: columna requerida {target} no encontrada; {list(df.columns)}")
        found[target]=col
    return df.rename(columns={v:k for k,v in found.items()})[list(found)]

# MJ + RIPS, concatenados sin deduplicar: un FUB en ambas hojas cuenta dos veces.
mj_alias={"fub":["FUB","fub"],"curp":["CURP","curp"],"fecha_hechos":["fechahechos","fecha_hechos"],"nombre":["Nombre","nombre"],"apellido_paterno":["Apellido Paterno","apellido_paterno"],"apellido_materno":["Apellido Materno","apellido_materno"],"estado":["ESTADO","estado"]}
mr_parts=[]
for tipo,hoja in {"MJ":"MJ UNICOS","RIPS":"RIPS UNICOS"}.items():
    tmp=leer_excel_map(MJ_RIPS_FILE,mj_alias,sheet=hoja);tmp["tipo"]=tipo;mr_parts.append(tmp)
mr=pd.concat(mr_parts,ignore_index=True);mr["fub"]=mr.fub.astype("string").str.strip().str.upper()
rn_status=u.rn[["fub_rnpdno","estatus_loc_derivada_rnpdno","is_cvid_derivada_rnpdno","is_svid_derivada_rnpdno"]].copy().drop_duplicates("fub_rnpdno")
rn_status.columns=["fub","estatus_loc","is_cvid","is_svid"];rn_status["fub"]=rn_status.fub.astype("string").str.strip().str.upper()
mr=mr.merge(rn_status,on="fub",how="left");mr["estatus_loc"]=mr.estatus_loc.fillna("des")
mr["is_cvid"]=mr.is_cvid.fillna(False).astype(bool);mr["is_svid"]=mr.is_svid.fillna(False).astype(bool)
mr_dfs={"reg_mr_min-max":mr,"reg_mr_min-max_des":mr[mr.estatus_loc.eq("des")],"reg_mr_min-max_loc":mr[mr.estatus_loc.eq("loc")]}
mr_dfs["reg_mr_min-max_loc_cvid"]=mr_dfs["reg_mr_min-max_loc"][mr_dfs["reg_mr_min-max_loc"].is_cvid]
mr_dfs["reg_mr_min-max_loc_svid"]=mr_dfs["reg_mr_min-max_loc"][mr_dfs["reg_mr_min-max_loc"].is_svid]
mr_vars={k:len(v) for k,v in mr_dfs.items()}
mr_den={"reg_mr_min-max_des":"reg_mr_min-max","reg_mr_min-max_loc":"reg_mr_min-max","reg_mr_min-max_loc_cvid":"reg_mr_min-max_loc","reg_mr_min-max_loc_svid":"reg_mr_min-max_loc"}
for k,d in mr_den.items():mr_vars[f"pct_{k}"]=f"{100*mr_vars[k]/mr_vars[d]:.1f}%" if mr_vars[d] else "n/d"
for f in ("dia_mes_anio","mes_anio","anio"):mr_vars[f"fha_mr_{f}"]=fecha_es("2026-08-20",f)

# Directorio congelado: los FUB sin cruce siguen en la categoría desaparecida.
dir_alias={"fub":["FUB","fub"],"estado_de_desaparicion":["Estado de Desaparición","estado_desaparicion","estado_de_desaparicion"]}
dir_df=leer_excel_map(DIRECTORIO_INDICIOS,dir_alias)
dir_df.fub=dir_df.fub.astype("string").str.upper();dir_df.estado_de_desaparicion=dir_df.estado_de_desaparicion.fillna("Sin información").replace({"México":"Estado de México"})
clas=u.rn[["fub_rnpdno","clasificacion_des_derivada_rnpdno"]].rename(columns={"fub_rnpdno":"fub","clasificacion_des_derivada_rnpdno":"clasificacion_des"}).copy();clas.fub=clas.fub.astype("string").str.upper()
dir_df=dir_df.merge(clas,on="fub",how="left")
es_loc=dir_df.clasificacion_des.eq("Localizados");es_des=dir_df.clasificacion_des.isin(["Desaparecidos","Pendientes"])|dir_df.clasificacion_des.isna()
def tab_estado(mask,col):return dir_df.loc[mask].value_counts("estado_de_desaparicion").rename_axis("estado_de_desaparicion").reset_index(name=col)
tabulado_fmz=tab_estado(es_des,"todavia_desaparecidos").merge(tab_estado(es_loc,"localizados"),on="estado_de_desaparicion",how="outer").fillna({"todavia_desaparecidos":0,"localizados":0})
tabulado_fmz[["todavia_desaparecidos","localizados"]]=tabulado_fmz[["todavia_desaparecidos","localizados"]].astype(int)
tabulado_fmz=tabulado_fmz.sort_values("todavia_desaparecidos",ascending=False)
renombres={"México":"Estado de México"};entidades32=[renombres.get(n,n) for k,n in ENTIDADES.items() if k<=32]
tabulado_fmz_32=tabulado_fmz[tabulado_fmz.estado_de_desaparicion.isin(entidades32)].reset_index(drop=True)
tabulado_fmz=tabulado_fmz[tabulado_fmz.estado_de_desaparicion.isin(CATALOGO_ENTIDADES_LAMINA)].sort_values("todavia_desaparecidos",ascending=False).reset_index(drop=True)
fmz_total=int(tabulado_fmz.localizados.sum());fmz_nac=int(es_loc.sum());fmz32=int(tabulado_fmz_32.localizados.sum())
fmz={"total":fmz_total,"pct":f"{100*fmz_total/fmz_nac:.1f}%" if fmz_nac else "n/d","nacional":fmz_nac,"resto":fmz_nac-fmz_total,"total_32":fmz32,"universo":int(tabulado_fmz.localizados.sum()+tabulado_fmz.todavia_desaparecidos.sum())}

# Resolver todos los marcadores, porcentajes, particiones y dos cortes de referencia.
vars_base=u.indicadores(CONTEOS,PORCENTAJES)
pct_particiones={k:v for suf,den,parts in PARTICIONES for k,v in porcentajes_particion(u,parts,den,suf).items()}
extra={**pct_particiones,**diferencias_vs_corte(vars_base,MAPA_MARZO),**deltas_vs_corte(u,SLICES_DELTA),
       **diferencias_vs_corte(vars_base,MAPA_JULIO),**deltas_vs_corte(u,SLICES_DELTA_JULIO,"_v2"),
       "reg_rn_24jun26-max_loc_fmz":fmz["total"],"pct_reg_rn_24jun26-max_loc_fmz":fmz["pct"],
       "reg_rn_24jun26-max_loc_fmz_v2":fmz["nacional"],"reg_rn_24jun26-max_loc_fmz_v3":fmz["resto"],
       "reg_rn_24jun26-max_loc_fmz_v4":fmz["total_32"],**mr_vars}
for periodo in ("01oct24-max","01ago26-max"):extra[f"fha_{periodo.split('-')[0]}_mes_anio"]=fecha_es(PERIODOS[periodo][0],"mes_anio")
vars_ppt=u.indicadores(CONTEOS,PORCENTAJES,extra)
print(f"Marcadores calculados: {len(vars_ppt)} | MJ/RIPS: {len(mr):,} | indicios formalizados: {fmz['total']:,}")

### Materializar bases y archivos de respaldo

In [ ]:
PARA_EXPORTAR=[Slice("rn","01oct24-max",estatus="des"),Slice("rn"),*[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("sisf","cisf_cind","cisf_sind")]]
exportados=u.dataframes(PARA_EXPORTAR)
for nombre,df_out in exportados.items():
    destino=RESULTADOS/f"{nombre}.parquet";df_out.to_parquet(destino,index=False,engine="pyarrow");print(f"💾 {nombre}: {len(df_out):,} → {destino.name}")

s_review=Slice("rn-no-ba","01jul26-max",estatus="des",subfiltro="sfval")
df_review=u.slice(s_review);token=FECHA_EMISION.strftime("%d%m%Y");nombre_review=f"{s_review.nombre}_{token}"
df_review.to_parquet(RESULTADOS/f"{nombre_review}.parquet",index=False,engine="pyarrow");df_review.to_excel(RESULTADOS/f"{nombre_review}.xlsx",index=False)
print(f"💾 {nombre_review}: {len(df_review):,} → Parquet / Excel")

# Bases `_dloc` y diferencia contra el corte: filas anteriores y actuales identificadas por FUB.
def exportar_bajas(s,corte,sufijo="",nombre=None):
    path,col=CORTES[corte]; real=_col_excel(path,col);prev_df=pd.read_excel(path,dtype={real:str});prev_df=prev_df.drop(columns=[c for c in prev_df if str(c).startswith("Unnamed:")]);prev_df["_fub"]=prev_df[real].astype("string").str.strip()
    actual=u.slice(s); ids={str(x).strip() for x in actual.fub_rnpdno.dropna() if str(x).strip()}; bajas=prev_df[prev_df._fub.notna() & ~prev_df._fub.isin(ids)].copy()
    ren={c:f"{c}{SUFIJOS_CORTE[corte]}" for c in bajas.columns if c!="_fub"};bajas=bajas.rename(columns=ren).drop(columns="_fub")
    keep=[c for c in ("fub_rnpdno","expedientes_rnpdno","estatus_loc_derivada_rnpdno","fecha_hechos_modificada_derivada_rnpdno",u.columna_bandera("rn",s.subfiltro.split("_")[0]) if s.subfiltro!="all" and s.subfiltro.split("_")[0] not in CONSTANTES["rn"] else None) if c and c in u.rn.columns]
    hoy=u.rn[keep].drop_duplicates("fub_rnpdno") if keep else pd.DataFrame()
    if not hoy.empty:bajas=bajas.merge(hoy,left_on=real+SUFIJOS_CORTE[corte],right_on="fub_rnpdno",how="left")
    return bajas

def exportar_movimientos(s,corte):
    ref,_=fubs_corte(corte,False);actual=u.slice(s);ids=actual.fub_rnpdno.astype("string").str.strip();altas=actual[ids.notna() & ids.ne("") & ~ids.isin(ref)].copy();bajas=exportar_bajas(s,corte)
    col=f"movimiento_derivada_rnpdno{SUFIJOS_CORTE[corte]}";altas[col]="alta";bajas[col]="baja";return pd.concat([altas,bajas],ignore_index=True,sort=False)[[col,*[c for c in pd.concat([altas,bajas],axis=0).columns if c!=col]]]

BAJAS_A_EXPORTAR=[(Slice("rn","01ene06-max",estatus="des",subfiltro="sisf"),"reg_rn_25mar26_des_sisf","_dloc")]
BAJAS_JULIO=[(Slice("rn","01ene06-max",estatus="des",subfiltro="sisf"),"reg_rn_01jul26_des_sisf","_dloc_v2")]
DIFERENCIAS_A_EXPORTAR=[(Slice("rn","01ene06-max",estatus="des",subfiltro="sisf"),"reg_rn_25mar26_des_sisf")]
for s,corte,suf in BAJAS_A_EXPORTAR+BAJAS_JULIO:
    df_b=exportar_bajas(s,corte,suf);dest=RESULTADOS/f"{s.nombre}{suf}.parquet";df_b.to_parquet(dest,index=False);print(f"💾 {dest.name}: {len(df_b):,}")
for s,corte in DIFERENCIAS_A_EXPORTAR:
    df_mov=exportar_movimientos(s,corte);dest=RESULTADOS/f"{s.nombre}-{corte}.parquet";df_mov.to_parquet(dest,index=False);print(f"💾 {dest.name}: {len(df_mov):,}")


## 6. Generar gráficas y la presentación

In [ ]:
def dimensiones_placeholder(pptx_path,clave):
    objetivo=f"{{{{{clave}}}}}";fact=914400/2.54;encontrados=[]
    def walk(shapes):
        for sh in shapes:
            if sh.shape_type==MSO_SHAPE_TYPE.GROUP:walk(sh.shapes)
            elif sh.has_text_frame and sh.text.strip()==objetivo:encontrados.append(sh)
    for slide in Presentation(str(pptx_path)).slides:walk(slide.shapes)
    if not encontrados:raise KeyError(f"Placeholder {objetivo} no encontrado en plantilla")
    return encontrados[0].width/fact,encontrados[0].height/fact

GRAFICAS=[];GRAFICAS_SIN_DATOS=[]
def _cargar_fuentes():
    for f in font_manager.findSystemFonts(fontpaths=[FUENTES_DIR]):
        try:font_manager.fontManager.addfont(f)
        except Exception:pass

def guardar_grafica(fig,nombre):
    fig.savefig(RUTA_GRAFICAS/f"{nombre}.png",dpi=300,transparent=True,bbox_inches="tight")
    fig.savefig(RUTA_GRAFICAS/f"{nombre}.svg",transparent=True,bbox_inches="tight")
    plt.close(fig);GRAFICAS.append(nombre)

def grafica_mensual(s,nombre,color,ancho=None,alto=None,apiladas=None):
    if apiladas is None:
        ini,fin=u.rango_fechas(s);tabla=agrupar_por_mes(u.slice(s),u.columna(s,"fecha"),ini,fin);cols=["registros"]
    else:
        series={label:agrupar_por_mes(u.slice(sl),u.columna(sl,"fecha"),*u.rango_fechas(sl)).set_index("mes").registros for label,sl in apiladas.items()}
        tabla=pd.concat(series,axis=1).fillna(0).astype(int).rename_axis("mes").reset_index();cols=list(series)
    if ancho is None:ancho,alto=dimensiones_placeholder(PLANTILLA,nombre)
    fig,ax=plt.subplots(figsize=(ancho/2.54,alto/2.54),dpi=300)
    pal=color if isinstance(color,list) else [color]
    if not tabla.empty:
        bottoms=np.zeros(len(tabla))
        for i,c in enumerate(cols):
            vals=tabla[c].to_numpy();bars=ax.bar(tabla.mes,bars_height if False else vals,bottom=bottoms,color=pal[i%len(pal)],width=24)
            if apiladas is not None:
                for bar,val in zip(bars,vals):
                    if val>0:ax.text(bar.get_x()+bar.get_width()/2,bottoms[list(bars).index(bar)]+val/2,f"{val:,}",ha="center",va="center",fontsize=7,color="white")
            bottoms+=vals
        step=max(1,len(tabla)//8);ax.set_xticks(tabla.mes.iloc[::step]);ax.set_xticklabels([x.strftime("%b-%y") for x in tabla.mes.iloc[::step]],rotation=0,fontsize=7)
        if apiladas is None:
            for bar,val in zip(bars,tabla[cols[0]]):
                if val:ax.text(bar.get_x()+bar.get_width()/2,val,f"{val:,}",ha="center",va="bottom",fontsize=6)
        else:
            for x,total in zip(tabla.mes,bottoms):ax.text(x,total,f"{total:,}",ha="center",va="bottom",fontsize=6)
    ax.grid(axis="y",alpha=.2);ax.set_axisbelow(True);ax.spines[["top","right"]].set_visible(False);ax.tick_params(axis="y",labelsize=8);fig.tight_layout()
    if tabla.empty or tabla.select_dtypes("number").to_numpy().sum()==0:GRAFICAS_SIN_DATOS.append(nombre);ax.clear();ax.text(.5,.5,"Sin registros en el período",ha="center",va="center",transform=ax.transAxes,color="#757575",fontsize=14);ax.axis("off")
    guardar_grafica(fig,nombre)

def tabulado_dependencia(s):
    d=u.slice(s);ent=u.columna(s,"entidad");return pd.DataFrame({"entidad":d[ent],"Fiscalía":d[u.columna_bandera(s,"fis")],"Comisión":d[u.columna_bandera(s,"com")],"Portal":d[u.columna_bandera(s,"por")]}).groupby("entidad",observed=True).sum().reset_index()

def grafica_horizontal(tabla,nombre,cols,colores):
    fig,ax=plt.subplots(figsize=(20/2.54,12/2.54),dpi=300)
    if not tabla.empty:
        y=np.arange(len(tabla));left=np.zeros(len(tabla))
        for c,color in zip(cols,colores):
            vals=tabla[c].fillna(0).to_numpy();bars=ax.barh(y,vals,left=left,color=color,label=c,height=.75)
            for yy,v,l in zip(y,vals,left):
                if v>0 and v>max(1,left.max()*.015):ax.text(l+v/2,yy,f"{v:,}",ha="center",va="center",fontsize=7,color="white")
            left+=vals
        ax.set_yticks(y);ax.set_yticklabels(tabla.iloc[:,0].astype(str),fontsize=8);ax.invert_yaxis()
        for yy,t in zip(y,left):ax.text(t,yy,f"{t:,.0f}",va="center",ha="left",fontsize=8)
    ax.grid(axis="x",alpha=.2);ax.set_axisbelow(True);ax.spines[["top","right","left"]].set_visible(False);ax.tick_params(axis="x",labelsize=8);fig.tight_layout()
    if tabla.empty or tabla.select_dtypes("number").to_numpy().sum()==0:GRAFICAS_SIN_DATOS.append(nombre);ax.clear();ax.text(.5,.5,"Sin registros en el período",ha="center",va="center",transform=ax.transAxes,color="#757575",fontsize=14);ax.axis("off")
    guardar_grafica(fig,nombre)

_cargar_fuentes()
# Las 6 gráficas mensuales del cuaderno original.
S_RN_BA=Slice("rn-si-ba","01oct24-max",estatus="des")
S_CF=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="cfval")
S_SF=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval")
S_SFV=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval_sfvac")
S_SFC=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval_cfvac")
grafica_mensual(S_RN_BA,"gr_mes_rn-si-ba_01oct24-max_des","#af5967ff")
grafica_mensual(S_CF,"gr_mes_rn-no-ba_01oct24-max_des_cfval","#9b2347ff")
grafica_mensual(S_SF,"gr_mes_rn-no-ba_01oct24-max_des_sfval","#595959ff")
grafica_mensual(S_SFV,"gr_mes_rn-no-ba_01oct24-max_des_sfval_sfvac","#757575ff")
grafica_mensual(S_SFC,"gr_mes_rn-no-ba_01oct24-max_des_sfval_cfvac","#888888ff")
grafica_mensual(S_RN_BA,"gr_mes_rn_01oct24-max_des_sisf-cind-sind",["#ed7d31ff","#2a7473ff","#56408fff"],apiladas={"Datos insuficientes":Slice("rn","01oct24-max",estatus="des",subfiltro="sisf"),"Con actividades":Slice("rn","01oct24-max",estatus="des",subfiltro="cisf_cind"),"Sin actividad":Slice("rn","01oct24-max",estatus="des",subfiltro="cisf_sind")})
# 4 gráficas por entidad/dependencia.
for s,n in [(S_CF,"gr_ent_rn-no-ba_01oct24-max_des_cfval"),(S_SFV,"gr_ent_rn-no-ba_01oct24-max_des_sfval_sfvac"),(Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro="cfval"),"gr_ent_rn-no-ba_01ago26-max_des_cfval"),(Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro="sfval_sfvac"),"gr_ent_rn-no-ba_01ago26-max_des_sfval_sfvac")]:grafica_horizontal(tabulado_dependencia(s),n,["Fiscalía","Comisión","Portal"],["#4f3130","#e44d03","#de639b"])
# Dos gráficas de indicios formalizados.
grafica_horizontal(tabulado_fmz,"gr_ent_rn_24jun26-max_loc_fmz",["todavia_desaparecidos","localizados"],["#32485f","#b3c5bf"])
grafica_horizontal(tabulado_fmz_32,"gr_32ent_rn_24jun26-max_loc_fmz",["todavia_desaparecidos","localizados"],["#32485f","#b3c5bf"])
print(f"Gráficas creadas: {len(GRAFICAS)}")

In [ ]:
class PPTReportBuilder:
    """Sustitución inline de variables de texto e imágenes de la plantilla."""
    def __init__(self,template):
        if not Path(template).is_file():raise FileNotFoundError(f"Plantilla inexistente: {template}")
        self.prs=Presentation(str(template));self.text={};self.images={}
    @staticmethod
    def format_value(key,value):
        if value is None:return ""
        if any(t in str(key).lower() for t in ("anio","año","year")):
            try:return str(int(float(value)))
            except (ValueError,TypeError):return str(value)
        if isinstance(value,str):return value
        try:
            n=float(value);return f"{n:,.0f}" if n.is_integer() else str(value)
        except (ValueError,TypeError):return str(value)
    def add_text_variables(self,values):
        self.text={str(k).strip():self.format_value(k,v) for k,v in values.items()};return self
    def add_image_variables(self,values):self.images={str(k).strip():str(v) for k,v in values.items()};return self
    def _paragraph(self,p):
        counts={k:0 for k in self.text}; full=p.text
        for k,v in self.text.items():
            placeholder=f"{{{{{k}}}}}"
            if placeholder in full:
                cnt=sum(r.text.count(placeholder) for r in p.runs)
                if cnt==0:cnt=full.count(placeholder)
                full=full.replace(placeholder,v);counts[k]=cnt
        if any(counts.values()) and p.runs:
            p.runs[0].text=full
            for r in p.runs[1:]:r.text=""
        return counts
    def _shape_text(self,shape):
        counts={k:0 for k in self.text}
        if shape.has_text_frame:
            for p in shape.text_frame.paragraphs:
                for k,v in self._paragraph(p).items():counts[k]+=v
        elif shape.has_table:
            for row in shape.table.rows:
                for cell in row.cells:
                    for p in cell.text_frame.paragraphs:
                        for k,v in self._paragraph(p).items():counts[k]+=v
        elif shape.shape_type==MSO_SHAPE_TYPE.GROUP:
            for sh in shape.shapes:
                for k,v in self._shape_text(sh).items():counts[k]+=v
        return counts
    def render(self,destino):
        txt={k:0 for k in self.text};imgs={k:0 for k in self.images}
        for slide in self.prs.slides:
            for sh in slide.shapes:
                for k,v in self._shape_text(sh).items():txt[k]+=v
            for sh in list(slide.shapes):
                if not sh.has_text_frame:continue
                for k,path in self.images.items():
                    if sh.text.strip()!=f"{{{{{k}}}}}":continue
                    if not Path(path).exists():print(f"⚠️ Imagen ausente: {path}");break
                    x,y,w,h=sh.left,sh.top,sh.width,sh.height;el=sh.element;el.getparent().remove(el);slide.shapes.add_picture(path,x,y,w,h);imgs[k]+=1;break
        destino=Path(destino);destino.parent.mkdir(parents=True,exist_ok=True);self.prs.save(str(destino))
        print(f"✅ Presentación guardada: {destino}")
        faltantes=[k for k,v in txt.items() if not v]
        if faltantes:print(f"⚠️ Marcadores sin coincidencia ({len(faltantes)}): {faltantes[:12]}")
        faltan_img=[k for k,v in imgs.items() if not v]
        if faltan_img:print(f"⚠️ Placeholders de imagen sin coincidencia: {faltan_img}")
        return {"output_path":destino,"text_report":txt,"image_report":imgs}

# Se inspecciona el resultado de entrada sin escribir al PPT todavía.
print("Plantilla:",PLANTILLA.name,"| marcadores de texto:",len(vars_ppt),"| imágenes:",len(GRAFICAS))

In [ ]:
builder=(PPTReportBuilder(PLANTILLA)
         .add_text_variables(vars_ppt)
         .add_image_variables({nombre:RUTA_GRAFICAS/f"{nombre}.png" for nombre in GRAFICAS}))
reporte=builder.render(SALIDA_PPT)

# Comprobaciones mínimas del entregable final.
assert SALIDA_PPT.is_file() and SALIDA_PPT.stat().st_size>0, "No se creó el PPTX final"
assert len(reporte["image_report"])==12, "No se registraron las 12 gráficas previstas"
imagenes_no_insertadas=[k for k,v in reporte["image_report"].items() if v==0]
if imagenes_no_insertadas:
    print("⚠️ Revisar placeholders de imágenes no reemplazados:", imagenes_no_insertadas)
print(f"Validación completada: PPTX {SALIDA_PPT.name} ({SALIDA_PPT.stat().st_size:,} bytes); "
      f"{sum(v>0 for v in reporte['text_report'].values())}/{len(vars_ppt)} marcadores de texto localizados; "
      f"{sum(v>0 for v in reporte['image_report'].values())}/{len(GRAFICAS)} gráficas insertadas.")

## 7. Validar y comprobar salidas

In [ ]:
# Verifica contratos de conteo y lectura de archivos de salida.
conteos_base_esperados={
 "reg_rn_min-max":412227,
 "reg_rn_01oct24-max_des":19965,
 "reg_rn_01ene06-max_des_sisf":38962,
 "reg_rn_01ene06-max_des_cisf_cind":38358,
 "reg_rn_01ene06-max_des_cisf_sind":50713,
}
for k,n in conteos_base_esperados.items():assert vars_ppt[k]==n,(k,vars_ppt[k],n)
parquets=[RESULTADOS/f"{s.nombre}.parquet" for s in PARA_EXPORTAR]
parquets += [RESULTADOS/f"{Slice('rn','01ene06-max',estatus='des',subfiltro='sisf').nombre}_dloc.parquet",
             RESULTADOS/f"{Slice('rn','01ene06-max',estatus='des',subfiltro='sisf').nombre}_dloc_v2.parquet",
             RESULTADOS/f"{Slice('rn','01ene06-max',estatus='des',subfiltro='sisf').nombre}-reg_rn_25mar26_des_sisf.parquet"]
assert all(p.is_file() for p in parquets),[str(p) for p in parquets if not p.is_file()]
assert all((RUTA_GRAFICAS/f"{n}.png").is_file() for n in GRAFICAS)
prs_check=Presentation(str(SALIDA_PPT))
assert len(prs_check.slides)==len(Presentation(str(PLANTILLA)).slides)
print(f"Validación correcta: {len(parquets)} bases Parquet, {len(GRAFICAS)} PNG y PPTX de {len(prs_check.slides)} láminas.")
print("Conteos de referencia verificados:",conteos_base_esperados)

# Resumen opcional por entidad, usando la misma definición del reporte.
resumen_estatal=pd.DataFrame([{"entidad":s.entidad_texto,"registros":u.contar(s)} for s in expandir_entidades(Slice("rn","01oct24-max",estatus="des"))]).sort_values("registros",ascending=False)
display(resumen_estatal.head(10))